In [0]:
#Completed with the help of Gemini and Databricks Assistant
from pyspark.sql import functions as F

volume_path = "/Volumes/cs483eviction/raw/milwaukee_foreclosures/"
target_table = "cs483eviction.cleaned.Milwaukee_cleaned"

#Loop through to create the SQL tables

# Loop through to create the SQL tables
for file in dbutils.fs.ls(volume_path):
    if file.name.endswith(".csv"):
        print(f"Processing: {file.name}")
        
        # 1. Read the CSV
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(file.path)
        )
        
        # 2. Clean column names (Delta/SQL tables don't allow spaces or special chars in headers)
        clean_cols = [
            c.strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_")
            for c in df.columns
        ]
        df = df.toDF(*clean_cols)
        
        # 3. Create a valid SQL table name from the filename
        table_name = ''.join(c if c.isalnum() else '_' for c in file.name.replace(".csv", "")).lower()
        
        # 4. Save as a SQL table in the 'raw' schema
        df.write.mode("overwrite").saveAsTable(f"cs483eviction.raw.{table_name}")
        print(f" -> Saved table: cs483eviction.raw.{table_name}")

In [0]:
tables = [f"cs483eviction.raw.foreclosuresgeocoded__{i}_" for i in range(1, 6)]

schemas = {t: spark.table(t).columns for t in tables}
all_cols = sorted(set().union(*schemas.values()))

for t, cols in schemas.items():
    missing = [c for c in all_cols if c not in cols]
    print(f"{t.split('.')[-1]} ({len(cols)} cols) | Missing: {missing}")

print("\nAll unique columns across all 5 tables:")
print(all_cols)

In [0]:
%sql
SELECT * FROM cs483eviction.raw.foreclosuresgeocoded__1_ LIMIT 10

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.raw.foreclosuresgeocoded__1_

In [0]:
%sql
SELECT * FROM cs483eviction.raw.foreclosuresgeocoded__2_ LIMIT 10

--MERGE INTO cs483eviction.foreclosuresgeocoded__1_ using the key aldermanic_2022

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.raw.foreclosuresgeocoded__2_

In [0]:
%sql
SELECT * FROM cs483eviction.raw.foreclosuresgeocoded__3_ LIMIT 10

--MERGE into cs483eviction.foreclosuresgeocoded__1_ using the key neighborhood

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.raw.foreclosuresgeocoded__3_

In [0]:
%sql
SELECT * FROM cs483eviction.raw.foreclosuresgeocoded__4_ LIMIT 10

--MERGE into cs483eviction.foreclosuresgeocoded__1_ using the key tract_2010

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.raw.foreclosuresgeocoded__4_

In [0]:
%sql
SELECT * FROM cs483eviction.raw.foreclosuresgeocoded__5_ LIMIT 10

--MERGE into cs483eviction.foreclosuresgeocoded__5_ on key tract_2020

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.raw.foreclosuresgeocoded__4_

In [0]:
%sql
CREATE OR REPLACE TABLE cs483eviction.cleaned.temp1 AS
SELECT 
  p.*,
  ald.privately_owned_parcels AS ald_parcels,
  ald.foreclosures AS ald_foreclosures,
  ald.foreclosures_per_1k AS ald_foreclosures_per_1k
FROM cs483eviction.raw.foreclosuresgeocoded__1_ p
INNER JOIN (
  SELECT 
    aldermanic_2022,
    MAX(privately_owned_parcels) AS privately_owned_parcels,
    MAX(foreclosures) AS foreclosures,
    MAX(foreclosures_per_1k) AS foreclosures_per_1k
  FROM cs483eviction.raw.foreclosuresgeocoded__2_
  WHERE aldermanic_2022 IS NOT NULL
  GROUP BY aldermanic_2022
) ald
  ON p.aldermanic_2022 = ald.aldermanic_2022;

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.cleaned.temp1;

In [0]:
%sql
SELECT * FROM cs483eviction.cleaned.temp1 LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TABLE cs483eviction.cleaned.temp2 AS
SELECT 
  t1.*,
  nbh.privately_owned_parcels AS nbh_parcels,
  nbh.foreclosures AS nbh_foreclosures,
  nbh.foreclosures_per_1k AS nbh_foreclosures_per_1k
FROM cs483eviction.cleaned.temp1 t1
LEFT JOIN (
  SELECT 
    neighborhood,
    MAX(privately_owned_parcels) AS privately_owned_parcels,
    MAX(foreclosures) AS foreclosures,
    MAX(foreclosures_per_1k) AS foreclosures_per_1k
  FROM cs483eviction.raw.foreclosuresgeocoded__3_
  WHERE neighborhood IS NOT NULL
  GROUP BY neighborhood
) nbh
  ON t1.neighborhood = nbh.neighborhood;
    


In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.cleaned.temp2;


In [0]:
%sql
SELECT * FROM cs483eviction.cleaned.temp2 LIMIT 10
    


In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT tract_2010) AS unique_tracts
FROM cs483eviction.raw.foreclosuresgeocoded__4_;

In [0]:
%sql
SELECT 
    start_year, 
    COUNT(*) as row_count 
FROM cs483eviction.raw.foreclosuresgeocoded__4_ 
GROUP BY start_year;

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT tract_2010) FROM cs483eviction.cleaned.temp2;

In [0]:
%sql
CREATE OR REPLACE TABLE cs483eviction.cleaned.temp3 AS
SELECT 
    t2.*,
    tr2010.start_year AS start_year,
    tr2010.privately_owned_parcels AS tr2010_parcels,
    tr2010.foreclosures AS tr2010_foreclosures,
    tr2010.foreclosures_per_1k AS tr2010_foreclosures_per_1k
FROM cs483eviction.cleaned.temp2 t2
LEFT JOIN (
    SELECT 
        TRIM(tract_2010) AS tract_2010_clean,
        MAX(start_year) AS start_year,
        MAX(privately_owned_parcels) AS privately_owned_parcels,
        MAX(foreclosures) AS foreclosures,
        MAX(foreclosures_per_1k) AS foreclosures_per_1k
    FROM cs483eviction.raw.foreclosuresgeocoded__4_
    WHERE tract_2010 IS NOT NULL AND start_year = 2010
    GROUP BY TRIM(tract_2010)
) tr2010
    ON TRIM(t2.tract_2010) = tr2010.tract_2010_clean;


In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.cleaned.temp3;

In [0]:
%sql
SELECT * FROM cs483eviction.cleaned.temp3 LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TABLE cs483eviction.cleaned.merged_df AS
SELECT 
    t3.*,
    tr2020.start_year AS start_year_2020,
    tr2020.privately_owned_parcels AS tr2020_parcels,
    tr2020.foreclosures AS tr2020_foreclosures,
    tr2020.foreclosures_per_1k AS tr2020_foreclosures_per_1k
FROM cs483eviction.cleaned.temp3 t3
LEFT JOIN (
    SELECT 
        TRIM(tract_2020) AS tract_2020_clean,
        MAX(start_year) AS start_year,
        MAX(privately_owned_parcels) AS privately_owned_parcels,
        MAX(foreclosures) AS foreclosures,
        MAX(foreclosures_per_1k) AS foreclosures_per_1k
    FROM cs483eviction.raw.foreclosuresgeocoded__5_
    WHERE tract_2020 IS NOT NULL AND start_year = 2020
    GROUP BY TRIM(tract_2020)
) tr2020
    ON TRIM(t3.tract_2020) = tr2020.tract_2020_clean;

In [0]:
%sql
SELECT COUNT(*) FROM cs483eviction.cleaned.merged_df;

In [0]:
%sql
SELECT * FROM cs483eviction.cleaned.merged_df LIMIT 10